# Control por Serial del firmware Eris (flavor `ErisServoDriver`)

Repo: https://github.com/JonathanCamargo/Eris

Este notebook habla el protocolo del flavor **`ErisServoDriver`** (16 canales de servo via PCA9685, pensado para ESP32/Teensy), que es **distinto** al `.ino` custom usado en `send_servo_command.ipynb`.

## Protocolo (segun el README del flavor)

- **Baud rate**: `921600` en ESP32 (limite del puente UART-USB integrado). Teensy y otras placas usan `115200`.
- Los comandos son **texto ASCII, separados por espacios, terminados en newline (`\n`)**.
- Comandos de servo (canales PCA9685, `0`-`15`, angulo `0`-`180`):
  - `X <canal> <angulo>` -> mueve **un** servo de forma inmediata.
  - `X <a0> <a1> ... <a15>` -> mueve **los 16 canales** de forma inmediata (16 valores).
  - `Y <canal> <angulo>` -> mueve un servo con **rampa suave** (rate-limited).
  - `Y <a0> <a1> ... <a15>` -> mueve los 16 canales con rampa suave.
- Otros comandos generales del firmware Eris:
  - `INFO` -> imprime info del firmware.
  - `ON` / `OFF` -> LED de la placa.
  - `S_F <feat1> <feat2> ...` -> configura que features se transmiten en streaming.
  - `S_ON` / `S_OFF` -> arranca/detiene el streaming.
  - `START` / `KILL` -> arranca/mata los hilos de sensores.

**Pendiente**: todavia falta la asignacion de canal PCA9685 -> eslabon/pata (hip/knee/ankle de cada pata). Dejo un diccionario `LEG_JOINT_CHANNELS` como placeholder mas abajo para completarlo despues.

In [19]:
import time
import serial
import serial.tools.list_ports
import ipywidgets as widgets
from IPython.display import display

for p in serial.tools.list_ports.comports():
    print(p.device, "-", p.description)

COM5 - Silicon Labs CP210x USB to UART Bridge (COM5)
COM3 - Standard Serial over Bluetooth link (COM3)
COM4 - Standard Serial over Bluetooth link (COM4)


Copia el puerto (ej. `COM5`) de la lista de arriba y ajustalo abajo antes de conectar.

In [20]:
PORT = "COM5"
BAUD = 921600  # ESP32 con ErisServoDriver (Teensy/otros: 115200)

NUM_CHANNELS = 16  # canales del PCA9685
ANGLE_MIN, ANGLE_MAX = 0, 180
NEUTRAL_ANGLE = 90  # posicion neutral de todos los actuadores

ser = serial.Serial(PORT, BAUD, timeout=2)
time.sleep(2)  # el ESP32 se reinicia al abrir el puerto serie
ser.reset_input_buffer()

boot_msg = ser.readline().decode(errors="replace").strip()
print("Board:", boot_msg if boot_msg else "(sin respuesta al abrir, puede ser normal)")

Board: (sin respuesta al abrir, puede ser normal)


In [21]:
import threading

# Mientras hay una captura INA activa, un hilo lector es el unico que lee el
# puerto; send_raw solo escribe para no robarle lineas del stream.
capture_reader_active = threading.Event()


def send_raw(command_text, read_response=True):
    """Manda una linea de texto tal cual (se agrega \n) y opcionalmente lee una linea de respuesta."""
    cmd = command_text.strip() + "\n"
    if capture_reader_active.is_set():
        ser.write(cmd.encode())
        return cmd.strip(), "(captura INA activa: respuesta omitida)"
    ser.reset_input_buffer()
    ser.write(cmd.encode())
    response = ser.readline().decode(errors="replace").strip() if read_response else ""
    return cmd.strip(), response


def _clamp_angle(angle):
    return max(ANGLE_MIN, min(ANGLE_MAX, angle))


def move_servo(channel, angle, smooth=False):
    """Mueve un solo canal (0-15) al angulo dado (0-180). smooth=True usa rampa (comando Y)."""
    if not (0 <= channel < NUM_CHANNELS):
        raise ValueError(f"canal fuera de rango 0-{NUM_CHANNELS - 1}: {channel}")
    cmd_letter = "Y" if smooth else "X"
    angle = _clamp_angle(angle)
    return send_raw(f"{cmd_letter} {channel} {angle:.2f}")


def move_all(angles, smooth=False):
    """Mueve los 16 canales, uno por comando.

    El firmware entiende tambien "X a0 a1 ... a15" en una sola linea, pero esa
    linea (~95 caracteres con 16 valores) excede el buffer de parseo del
    SerialCommand del firmware y se trunca, causando el error
    "expected 2 or 16 arguments" aunque se hayan mandado los 16 valores.
    Por eso aca se manda un "X <canal> <angulo>" corto por canal.
    Devuelve una lista de tuplas (comando_enviado, respuesta), una por canal.
    """
    if len(angles) != NUM_CHANNELS:
        raise ValueError(f"se esperaban {NUM_CHANNELS} angulos, llegaron {len(angles)}")
    return [move_servo(ch, a, smooth=smooth) for ch, a in enumerate(angles)]

## Asignacion de canal PCA9685 por pata/eslabon

- Delantera derecha (FR): hip=0, knee=1, ankle=2
- Delantera izquierda (FL): hip=4, knee=5, ankle=6
- Trasera derecha (RR): hip=8, knee=9, ankle=10
- Trasera izquierda (RL): hip=12, knee=13, ankle=14

In [ ]:
LEG_JOINT_CHANNELS = {
    "FR": {"hip": 0, "knee": 1, "ankle": 2},
    "FL": {"hip": 4, "knee": 5, "ankle": 6},
    "RR": {"hip": 8, "knee": 9, "ankle": 10},
    "RL": {"hip": 12, "knee": 13, "ankle": 14},
}

# Convencion de "angulo biomecanico": 0 = reposo (servo horizontal, raw=90),
# positivo = eslabon hacia arriba (visto desde afuera del robot), igual para
# las 4 patas. Calibrado a mano moviendo cada canal desde 90 (ver bitacora):
# en FR/FL, hip y ankle suben al subir el angulo crudo (mismo sentido que
# "positivo" = signo +1), pero la rodilla queda invertida (signo -1). En
# RR/RL es exactamente al reves (hip/ankle signo -1, rodilla signo +1) porque
# esas patas estan armadas espejadas respecto a las delanteras (mismo tipo de
# servo y eslabon, montaje rotado 180 grados de adelante hacia atras).
_FRONT_JOINT_SIGN = {"hip": 1, "knee": -1, "ankle": 1}
JOINT_SIGN = {
    "FR": dict(_FRONT_JOINT_SIGN),
    "FL": dict(_FRONT_JOINT_SIGN),
    "RR": {j: -s for j, s in _FRONT_JOINT_SIGN.items()},
    "RL": {j: -s for j, s in _FRONT_JOINT_SIGN.items()},
}


def bio_to_raw_angle(leg, joint, bio_angle):
    """Angulo biomecanico (0 = reposo, +arriba) -> angulo crudo de servo (0-180, 90 = reposo)."""
    sign = JOINT_SIGN[leg][joint]
    return _clamp_angle(NEUTRAL_ANGLE + sign * bio_angle)


def raw_to_bio_angle(leg, joint, raw_angle):
    """Angulo crudo de servo (0-180) -> angulo biomecanico (0 = reposo, +arriba)."""
    sign = JOINT_SIGN[leg][joint]
    return sign * (raw_angle - NEUTRAL_ANGLE)


def move_joint(leg, joint, bio_angle, smooth=False):
    """Mueve un eslabon a un angulo BIOMECANICO (0 = reposo, servo horizontal;
    positivo = eslabon hacia arriba, igual sentido en las 4 patas).

    Convierte internamente al angulo crudo del servo segun JOINT_SIGN antes de
    mandar el comando (ver bio_to_raw_angle). Para mandar el angulo crudo tal
    cual (sin esta correccion), usar move_servo(channel, angle_crudo).
    """
    channel = LEG_JOINT_CHANNELS[leg][joint]
    if channel is None:
        raise ValueError(f"todavia no se asigno un canal para {leg}/{joint} en LEG_JOINT_CHANNELS")
    raw_angle = bio_to_raw_angle(leg, joint, bio_angle)
    return move_servo(channel, raw_angle, smooth=smooth)

### Ir a posicion neutral (90 en los 16 canales)

Los sliders de abajo arrancan dibujados en 90, pero eso no mueve al servo hasta que lo arrastras. Este boton manda el comando de verdad.

In [23]:
neutral_button = widgets.Button(description="Ir a posicion neutral (90)", button_style="warning")
neutral_output = widgets.Output()


def on_neutral_clicked(_):
    with neutral_output:
        neutral_output.clear_output(wait=True)
        results = move_all([NEUTRAL_ANGLE] * NUM_CHANNELS, smooth=True)
        for sent, response in results:
            print(f"> {sent}  < {response if response else '(sin respuesta)'}")


neutral_button.on_click(on_neutral_clicked)
display(neutral_button, neutral_output)

Button(button_style='warning', description='Ir a posicion neutral (90)', style=ButtonStyle())

Output()

## Opcion A: comando crudo (igual que por el Monitor Serial)

Escribe el comando tal cual lo espera el firmware, por ejemplo `X 0 90` o `Y 1 30`.

In [ ]:
command_box = widgets.Text(value="X 0 90", description="Comando:", layout=widgets.Layout(width="300px"))
send_button = widgets.Button(description="Enviar", button_style="primary")
output_box = widgets.Output()


def on_send_clicked(_):
    with output_box:
        sent, response = send_raw(command_box.value)
        print(f"> {sent}")
        print(f"< {response if response else '(sin respuesta)'}")


send_button.on_click(on_send_clicked)
display(widgets.HBox([command_box, send_button]), output_box)

## Opcion B: un slider por canal (0-15)

Cada slider manda su propio comando `X <canal> <angulo>` (inmediato) al soltar.

In [25]:
smooth_checkbox = widgets.Checkbox(value=False, description="Rampa suave (Y en vez de X)")
channel_output = widgets.Output()


def make_on_change(channel):
    def on_change(change):
        if change["name"] != "value":
            return
        with channel_output:
            channel_output.clear_output(wait=True)
            sent, response = move_servo(channel, change["new"], smooth=smooth_checkbox.value)
            print(f"> {sent}")
            print(f"< {response if response else '(sin respuesta)'}")
    return on_change


channel_sliders = []
for ch in range(NUM_CHANNELS):
    slider = widgets.IntSlider(value=NEUTRAL_ANGLE, min=ANGLE_MIN, max=ANGLE_MAX, description=f"Ch {ch}", continuous_update=False)
    slider.observe(make_on_change(ch))
    channel_sliders.append(slider)

display(smooth_checkbox, widgets.GridBox(channel_sliders, layout=widgets.Layout(grid_template_columns="repeat(4, 220px)")), channel_output)

Checkbox(value=False, description='Rampa suave (Y en vez de X)')

GridBox(children=(IntSlider(value=90, continuous_update=False, description='Ch 0', max=180), IntSlider(value=9…

Output()

## Opcion C: sliders por pata/eslabon

Usa `LEG_JOINT_CHANNELS` para mostrar un slider por `hip`/`knee`/`ankle` de cada pata, en vez de por canal crudo.

**Estos sliders son en angulo BIOMECANICO** (0 = reposo, servo horizontal; +90/-90 = eslabon hacia arriba/abajo, mismo sentido en las 4 patas), no el angulo crudo del servo. `move_joint()` convierte internamente al angulo crudo segun `JOINT_SIGN` (ver celda de `LEG_JOINT_CHANNELS`), que compensa que las patas traseras estan armadas espejadas respecto a las delanteras. Para mandar el angulo crudo del servo tal cual, usar la Opcion A o B.

In [ ]:
leg_smooth_checkbox = widgets.Checkbox(value=False, description="Rampa suave (Y en vez de X)")
leg_output = widgets.Output()

BIO_ANGLE_RANGE = 90  # rango simetrico +-90 alrededor del reposo (0), ver JOINT_SIGN


def make_joint_on_change(leg, joint):
    def on_change(change):
        if change["name"] != "value":
            return
        with leg_output:
            leg_output.clear_output(wait=True)
            sent, response = move_joint(leg, joint, change["new"], smooth=leg_smooth_checkbox.value)
            print(f"> {sent}")
            print(f"< {response if response else '(sin respuesta)'}")
    return on_change


leg_panels = []
for leg, joints in LEG_JOINT_CHANNELS.items():
    joint_sliders = []
    for joint, channel in joints.items():
        slider = widgets.IntSlider(
            value=0, min=-BIO_ANGLE_RANGE, max=BIO_ANGLE_RANGE,
            description=f"{joint} (ch{channel})", continuous_update=False,
            style={"description_width": "90px"},
        )
        slider.observe(make_joint_on_change(leg, joint))
        joint_sliders.append(slider)
    leg_panels.append(widgets.VBox([widgets.Label(f"Pata {leg}")] + joint_sliders))

display(leg_smooth_checkbox, widgets.HBox(leg_panels), leg_output)

## Opcion D: enviar trayectoria desde CSV

Envia un CSV con la trayectoria de angulos **ya mapeados por actuador** (uno o mas de los 12 servos de `LEG_JOINT_CHANNELS`, o canales crudos del PCA9685).

Formato esperado del CSV:

- Columna de tiempo opcional, primer nombre que coincida entre `t`, `t_s`, `time`, `time_s` (segundos) o `t_ms` (milisegundos). Si esta presente, la reproduccion respeta esos intervalos (tiempo real de la trayectoria grabada/planeada). Si no esta, se usa un paso fijo `1 / Hz` (campo "Hz" del panel de abajo).
- Una columna por actuador:
  - `FR_hip`, `FR_knee`, `FR_ankle`, `FL_hip`, `FL_knee`, `FL_ankle`, `RR_hip`, `RR_knee`, `RR_ankle`, `RL_hip`, `RL_knee`, `RL_ankle` (segun `LEG_JOINT_CHANNELS`): angulo **BIOMECANICO** (0 = reposo/horizontal, +90/-90 = eslabon arriba/abajo, mismo sentido en las 4 patas). Se convierten a angulo crudo de servo con `JOINT_SIGN` antes de enviar (compensa que las patas traseras estan espejadas respecto a las delanteras).
  - `ch0` ... `ch15` para canales crudos del PCA9685 (0-180, sin conversion): util para los canales 3/7/11/15, que no tienen eslabon asignado.
- No hace falta incluir las 16 columnas: solo se envian las columnas de actuador presentes en el CSV.
- **Pata**: el dropdown "Pata" filtra el envio a los canales de esa pata (hip/knee/ankle) aunque el CSV traiga columnas de otras patas; con "Todas (segun CSV)" se manda cualquier columna de actuador reconocida.

Ejemplo minimo (2 filas, 3 actuadores en angulo biomecanico, con tiempo en segundos):

```
t,FR_hip,FR_knee,FR_ankle
0.0,0,0,0
0.5,15,-10,5
```

Mientras hay una toma INA activa (Opcion de mas abajo) las respuestas del firmware se omiten automaticamente para no pelear por el puerto, igual que con los demas comandos.

In [ ]:
import csv as _csv

TRAJ_TIME_COLUMNS = ("t", "t_s", "time", "time_s", "t_ms")

# Alias de nombre de pata -> clave de LEG_JOINT_CHANNELS. Acepta tanto la
# convencion FR/FL/RR/RL como fr/fl/br/bl (front/back right/left), en
# mayusculas o minusculas.
_LEG_ALIASES = {
    "fr": "FR", "fl": "FL",
    "rr": "RR", "rl": "RL",
    "br": "RR", "bl": "RL",
}


def _resolve_actuator_columns(fieldnames):
    """Mapea columnas del CSV a (canal_pca9685, signo_biomecanico).

    Acepta "<pata>_<eslabon>" con o sin sufijo "_joint" (ej. "FR_hip",
    "fr_hip_joint"), en cualquier combinacion de mayus/minus, usando
    _LEG_ALIASES para la pata: esas columnas se interpretan como angulo
    BIOMECANICO (0 = reposo, ver JOINT_SIGN) y se convierten a angulo crudo
    antes de enviar. "ch<N>" (canal crudo 0-15) se manda tal cual, sin
    conversion (signo None): util para los canales sin eslabon asignado
    (3/7/11/15) o para mandar angulo crudo directo.
    Ignora las demas columnas (p.ej. la de tiempo).
    """
    columns = {}
    for name in fieldnames or []:
        if name is None:
            continue
        key = name.strip().lower()
        if key.endswith("_joint"):
            key = key[: -len("_joint")]
        leg_alias, _, joint = key.partition("_")
        leg = _LEG_ALIASES.get(leg_alias)
        if leg is not None and joint in LEG_JOINT_CHANNELS[leg]:
            columns[name] = (LEG_JOINT_CHANNELS[leg][joint], JOINT_SIGN[leg][joint])
            continue
        if key.startswith("ch") and key[2:].isdigit():
            ch = int(key[2:])
            if 0 <= ch < NUM_CHANNELS:
                columns[name] = (ch, None)
    return columns


def load_trajectory_csv(path):
    """Lee un CSV de trayectoria.

    Devuelve (time_col, actuator_columns, rows):
      - time_col: nombre de la columna de tiempo encontrada (o None).
      - actuator_columns: dict {nombre_columna: (canal_pca9685, signo_o_None)}.
      - rows: lista de dicts, una por fila del CSV (valores como texto).
    """
    with open(path, newline="", encoding="utf-8") as f:
        reader = _csv.DictReader(f)
        rows = list(reader)
        fieldnames = reader.fieldnames or []
    time_col = next((c for c in TRAJ_TIME_COLUMNS if c in fieldnames), None)
    actuator_columns = _resolve_actuator_columns(fieldnames)
    if not actuator_columns:
        raise ValueError(
            "no se encontraron columnas de actuador reconocibles en el CSV "
            "(ej. FR_hip, fr_hip_joint, ch0); columnas presentes: " + ", ".join(fieldnames)
        )
    return time_col, actuator_columns, rows


def send_trajectory(path, smooth=False, rate_hz=50.0, read_response=True, stop_event=None, on_row=None, channels=None):
    """Reproduce un CSV de trayectoria mandando cada fila al firmware por Serial.

    - Si el CSV trae columna de tiempo (t/t_s/time/time_s/t_ms), respeta esos
      intervalos (tiempo real de la trayectoria). Si no, usa paso fijo 1/rate_hz.
    - Las columnas "<pata>_<eslabon>" traen angulo BIOMECANICO (0 = reposo);
      se convierten a angulo crudo de servo con JOINT_SIGN antes de mandar.
      Las columnas "ch<N>" se mandan tal cual (crudo), sin conversion.
    - smooth=True usa el comando Y (rampa) en vez de X (inmediato).
    - read_response=True lee la respuesta "OK ..." de cada comando (mas lento
      pero deja ver errores); se desactiva solo si hay una toma INA activa, para
      no competir por el puerto con el hilo lector de esa captura.
    - stop_event: threading.Event opcional para cortar la reproduccion a mitad.
    - on_row(i, total, row): callback opcional de progreso.
    - channels: iterable opcional de canales PCA9685 (0-15) a los que restringir
      el envio (p.ej. LEG_JOINT_CHANNELS["FR"].values()); las columnas del CSV
      que mapean a otros canales se ignoran, para mandarle la trayectoria a una
      sola pata sin tocar las demas aunque el CSV traiga las 12/16 columnas.
    """
    time_col, actuator_columns, rows = load_trajectory_csv(path)
    if channels is not None:
        channel_set = set(channels)
        actuator_columns = {col: (ch, sign) for col, (ch, sign) in actuator_columns.items() if ch in channel_set}
        if not actuator_columns:
            raise ValueError("ninguna columna del CSV corresponde a los canales seleccionados")
    if not rows:
        print("CSV vacio, nada que enviar.")
        return
    cmd_letter = "Y" if smooth else "X"
    total = len(rows)
    t0 = time.monotonic()
    can_read = read_response and not capture_reader_active.is_set()
    for i, row in enumerate(rows):
        if stop_event is not None and stop_event.is_set():
            print(f"Trayectoria detenida en la fila {i}/{total}.")
            return
        if time_col is not None:
            t_row = float(row[time_col])
            if time_col == "t_ms":
                t_row /= 1000.0
            delay = (t0 + t_row) - time.monotonic()
            if delay > 0:
                time.sleep(delay)
        elif i > 0:
            time.sleep(1.0 / rate_hz)
        for col, (channel, sign) in actuator_columns.items():
            raw = row.get(col)
            if raw is None or raw == "":
                continue
            value = float(raw)
            angle = value if sign is None else NEUTRAL_ANGLE + sign * value
            cmd = f"{cmd_letter} {channel} {_clamp_angle(angle):.2f}\n"
            ser.write(cmd.encode())
            if can_read:
                ser.readline()
        if not can_read:
            ser.reset_input_buffer()
        if on_row is not None:
            on_row(i, total, row)
    print(f"Trayectoria enviada: {total} filas x {len(actuator_columns)} actuadores ({', '.join(actuator_columns)}).")

In [ ]:
import tkinter as _tk
from tkinter import filedialog as _filedialog

traj_path_box = widgets.Text(value="", description="CSV:", disabled=True, layout=widgets.Layout(width="420px"))
traj_pick_button = widgets.Button(description="Elegir archivo...")
traj_leg_dropdown = widgets.Dropdown(
    options=["Todas (segun CSV)"] + list(LEG_JOINT_CHANNELS),
    value="Todas (segun CSV)",
    description="Pata:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)
traj_smooth_checkbox = widgets.Checkbox(value=False, description="Rampa suave (Y en vez de X)")
traj_rate_box = widgets.FloatText(value=50.0, description="Hz (sin columna de tiempo):", style={"description_width": "200px"}, layout=widgets.Layout(width="320px"))
traj_send_button = widgets.Button(description="Enviar trayectoria", button_style="success")
traj_stop_button = widgets.Button(description="Detener", button_style="danger", disabled=True)
traj_progress = widgets.IntProgress(min=0, max=1, value=0, layout=widgets.Layout(width="420px"))
traj_output = widgets.Output()

_traj_state = {"stop_event": None, "thread": None}


def _pick_trajectory_file(_):
    root = _tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    path = _filedialog.askopenfilename(parent=root, title="Elegir CSV de trayectoria", filetypes=[("CSV", "*.csv"), ("Todos", "*.*")])
    root.destroy()
    if path:
        traj_path_box.value = path


def _on_traj_row(i, total, row):
    traj_progress.max = max(total, 1)
    traj_progress.value = i + 1


def _send_trajectory_thread(path, smooth, rate_hz, stop_event, channels):
    with traj_output:
        try:
            send_trajectory(path, smooth=smooth, rate_hz=rate_hz, stop_event=stop_event, on_row=_on_traj_row, channels=channels)
        except Exception as exc:
            print(f"Error: {exc}")
    traj_send_button.disabled = False
    traj_stop_button.disabled = True


def _on_traj_send_clicked(_):
    path = traj_path_box.value.strip()
    leg = traj_leg_dropdown.value
    channels = None if leg == "Todas (segun CSV)" else list(LEG_JOINT_CHANNELS[leg].values())
    with traj_output:
        traj_output.clear_output(wait=True)
        if not path:
            print("Elegi primero un archivo CSV.")
            return
        if ser is None or not ser.is_open:
            print("El puerto serie no esta abierto. Ejecuta la celda de conexion.")
            return
        print(f"Enviando trayectoria: {path}" + (f" (solo pata {leg}, canales {channels})" if channels is not None else ""))
    stop_event = threading.Event()
    _traj_state["stop_event"] = stop_event
    traj_send_button.disabled = True
    traj_stop_button.disabled = False
    traj_progress.value = 0
    thread = threading.Thread(
        target=_send_trajectory_thread,
        args=(path, traj_smooth_checkbox.value, traj_rate_box.value, stop_event, channels),
        daemon=True,
    )
    _traj_state["thread"] = thread
    thread.start()


def _on_traj_stop_clicked(_):
    if _traj_state["stop_event"] is not None:
        _traj_state["stop_event"].set()


traj_pick_button.on_click(_pick_trajectory_file)
traj_send_button.on_click(_on_traj_send_clicked)
traj_stop_button.on_click(_on_traj_stop_clicked)

display(
    widgets.HBox([traj_path_box, traj_pick_button]),
    widgets.HBox([traj_leg_dropdown, traj_smooth_checkbox, traj_rate_box]),
    widgets.HBox([traj_send_button, traj_stop_button]),
    traj_progress,
    traj_output,
)

## Medicion de corriente (INA219): ventana emergente

**Requiere el firmware `eris_servo_ina/eris_servo_ina.ino`** (ErisServoDriver + INA219), no el `ErisServoDriver` original: ese no tiene soporte de INA219. El INA219 va conectado directo al ESP32 en un bus I2C propio (`Wire1`: SDA=GPIO32, SCL=GPIO33 por defecto, editable en el `.ino`), separado del PCA9685.

**Requiere tambien `matplotlib`** (`pip install matplotlib`) para el grafico en vivo.

La celda de abajo abre una ventana (tkinter) con:

- selector de pata y eslabon (12 servos, **una toma por servo**) y tabla con el estado de cada toma,
- **grafico en tiempo real** de corriente (mA), voltaje (V) y potencia (mW) mientras se captura, al lado de la tabla; al seleccionar una fila de una toma ya terminada (sin capturar) se muestra su curva completa,
- **Iniciar** / **Detener**: manda `S_TIME`, `S_F INA`, `S_ON` / `S_OFF` y guarda las lineas `D <t_ms> <mA> <V> <mW>` (100 Hz),
- **Exportar seleccionado** (CSV de un servo) y **Exportar todo** (un CSV con columnas `leg,joint,channel`).

Mientras hay una toma activa se pueden seguir moviendo los servos con los widgets de arriba (para excitar el servo elegido); sus respuestas no se muestran porque el hilo lector usa el puerto.

In [ ]:
import csv
import queue
import tkinter as tk
from collections import deque
from tkinter import ttk, filedialog, messagebox

from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

INA_COLUMNS = ("t_ms", "current_mA", "bus_V", "power_mW")
SERVO_SLOTS = [(leg, joint, ch) for leg, joints in LEG_JOINT_CHANNELS.items() for joint, ch in joints.items()]
ina_captures = {}  # (leg, joint) -> lista de tuplas (t_ms, current_mA, bus_V, power_mW)
PLOT_BUFFER_LEN = 3000  # puntos maximos en el grafico en vivo (rolling buffer)
_ina_popup_thread = None


def _ina_reader_loop(stop_event, out_queue):
    buf = b""
    while not stop_event.is_set():
        try:
            n = ser.in_waiting
            if not n:
                time.sleep(0.005)
                continue
            buf += ser.read(n)
        except serial.SerialException as exc:
            out_queue.put(("L", f"error de puerto: {exc}"))
            return
        *lines, buf = buf.split(b"\n")
        for raw in lines:
            text = raw.decode(errors="replace").strip()
            if text.startswith("D "):
                try:
                    values = tuple(float(x) for x in text.split()[1:])
                    if len(values) == len(INA_COLUMNS):
                        out_queue.put(("D", values))
                        continue
                except ValueError:
                    pass
            if text:
                out_queue.put(("L", text))


def _run_ina_popup():
    root = tk.Tk()
    root.title("Captura de corriente INA219")
    root.geometry("1150x620")
    root.report_callback_exception = lambda exc, val, tb: messagebox.showerror("Error", str(val))
    root.attributes("-topmost", True)
    root.after(300, lambda: root.attributes("-topmost", False))

    state = {"capturing": False, "slot": None, "stop": None, "thread": None, "queue": None}
    plot_buf = {"t": deque(maxlen=PLOT_BUFFER_LEN), "i": deque(maxlen=PLOT_BUFFER_LEN),
                "v": deque(maxlen=PLOT_BUFFER_LEN), "p": deque(maxlen=PLOT_BUFFER_LEN)}
    leg_var = tk.StringVar(value=SERVO_SLOTS[0][0])
    joint_var = tk.StringVar(value=SERVO_SLOTS[0][1])
    status_var = tk.StringVar(value="Listo. Elegi pata y eslabon, y presiona Iniciar.")
    last_var = tk.StringVar(value="Ultima lectura: -")

    top = ttk.Frame(root, padding=10)
    top.pack(side="top", fill="x")
    ttk.Label(top, text="Pata:").pack(side="left")
    ttk.Combobox(top, textvariable=leg_var, values=list(LEG_JOINT_CHANNELS), width=5, state="readonly").pack(side="left", padx=(4, 12))
    ttk.Label(top, text="Eslabon:").pack(side="left")
    ttk.Combobox(top, textvariable=joint_var, values=["hip", "knee", "ankle"], width=8, state="readonly").pack(side="left", padx=(4, 12))
    start_btn = ttk.Button(top, text="Iniciar")
    stop_btn = ttk.Button(top, text="Detener", state="disabled")
    start_btn.pack(side="left", padx=4)
    stop_btn.pack(side="left", padx=4)

    # El frame de botones de exportar y las etiquetas de estado se anclan al
    # fondo (side="bottom") ANTES de empacar el panel con expand=True, para
    # que siempre queden visibles aunque la ventana se achique (antes, con
    # geometry fija chica, el boton "Exportar" podia quedar recortado fuera
    # de la ventana).
    bottom = ttk.Frame(root, padding=10)
    bottom.pack(side="bottom", fill="x")
    export_sel_btn = ttk.Button(bottom, text="Exportar seleccionado")
    export_all_btn = ttk.Button(bottom, text="Exportar todo")
    export_sel_btn.pack(side="left", padx=4)
    export_all_btn.pack(side="left", padx=4)

    ttk.Label(root, textvariable=status_var).pack(side="bottom", anchor="w", padx=10)
    ttk.Label(root, textvariable=last_var).pack(side="bottom", anchor="w", padx=10)

    paned = ttk.PanedWindow(root, orient="horizontal")
    paned.pack(side="top", fill="both", expand=True, padx=10, pady=4)

    tree_frame = ttk.Frame(paned)
    columns = ("leg", "joint", "channel", "samples", "dur_s", "mean_mA", "peak_mA")
    headers = ("Pata", "Eslabon", "Canal", "Muestras", "Duracion (s)", "Media (mA)", "Pico (mA)")
    tree = ttk.Treeview(tree_frame, columns=columns, show="headings", height=12, selectmode="browse")
    for col, head in zip(columns, headers):
        tree.heading(col, text=head)
        tree.column(col, width=90, anchor="center")
    tree.pack(fill="both", expand=True)
    for leg, joint, ch in SERVO_SLOTS:
        tree.insert("", "end", iid=f"{leg}/{joint}", values=(leg, joint, ch, 0, "-", "-", "-"))
    paned.add(tree_frame, weight=1)

    plot_frame = ttk.Frame(paned)
    fig = Figure(figsize=(5.0, 4.2), dpi=100)
    ax_i, ax_v, ax_p = fig.subplots(3, 1, sharex=True)
    line_i, = ax_i.plot([], [], color="tab:red")
    line_v, = ax_v.plot([], [], color="tab:blue")
    line_p, = ax_p.plot([], [], color="tab:green")
    ax_i.set_ylabel("I (mA)")
    ax_v.set_ylabel("V (V)")
    ax_p.set_ylabel("P (mW)")
    ax_p.set_xlabel("t (s)")
    for ax in (ax_i, ax_v, ax_p):
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    canvas = FigureCanvasTkAgg(fig, master=plot_frame)
    canvas.get_tk_widget().pack(fill="both", expand=True)
    paned.add(plot_frame, weight=1)

    def current_slot():
        leg, joint = leg_var.get(), joint_var.get()
        channel = LEG_JOINT_CHANNELS[leg][joint]
        return (leg, joint), channel

    def refresh_row(slot):
        rows = ina_captures.get(slot, [])
        channel = LEG_JOINT_CHANNELS[slot[0]][slot[1]]
        if rows:
            currents = [r[1] for r in rows]
            values = (slot[0], slot[1], channel, len(rows), f"{(rows[-1][0] - rows[0][0]) / 1000:.2f}",
                      f"{sum(currents) / len(currents):.1f}", f"{max(currents):.1f}")
        else:
            values = (slot[0], slot[1], channel, 0, "-", "-", "-")
        tree.item(f"{slot[0]}/{slot[1]}", values=values)

    def _autoscale_nonnegative(ax, ys, pad_frac=0.1):
        """Autoscale un eje que siempre deberia ser >= 0 (mA, mW): fuerza el piso
        en 0 (referencia fija) y ajusta el techo al pico visible con margen, en
        vez de depender del margen por defecto de autoscale_view (que puede
        quedar minusculo, ej. -0.05..0.05, cuando los valores son chicos)."""
        peak = max(ys) if ys else 0.0
        top = peak * (1 + pad_frac) if peak > 0 else 1.0
        ax.set_ylim(0, top)

    def update_plot(xs, ys_i, ys_v, ys_p, title=""):
        line_i.set_data(xs, ys_i)
        line_v.set_data(xs, ys_v)
        line_p.set_data(xs, ys_p)
        _autoscale_nonnegative(ax_i, ys_i)
        _autoscale_nonnegative(ax_p, ys_p)
        ax_v.relim()
        ax_v.autoscale_view()
        ax_i.set_title(title, fontsize=9)
        canvas.draw_idle()

    def show_stored_plot(slot):
        rows = ina_captures.get(slot, [])
        if not rows:
            update_plot([], [], [], [], title=f"{slot[0]} {slot[1]}: sin datos")
            return
        t0 = rows[0][0]
        xs = [(r[0] - t0) / 1000.0 for r in rows]
        update_plot(xs, [r[1] for r in rows], [r[2] for r in rows], [r[3] for r in rows],
                    title=f"{slot[0]} {slot[1]} ({len(rows)} muestras)")

    def on_tree_select(_):
        selected = tree.selection()
        if selected and not state["capturing"]:
            leg, joint = selected[0].split("/")
            leg_var.set(leg)
            joint_var.set(joint)
            show_stored_plot((leg, joint))

    tree.bind("<<TreeviewSelect>>", on_tree_select)

    def start():
        if state["capturing"]:
            return
        if ser is None or not ser.is_open:
            messagebox.showerror("Puerto", "El puerto serie no esta abierto. Ejecuta la celda de conexion.")
            return
        slot, channel = current_slot()
        if ina_captures.get(slot) and not messagebox.askyesno("Sobrescribir", f"Ya hay una toma de {slot[0]} {slot[1]}. Sobrescribir?"):
            return
        ina_captures[slot] = []
        for buf in plot_buf.values():
            buf.clear()
        update_plot([], [], [], [], title=f"{slot[0]} {slot[1]} (en vivo)")
        refresh_row(slot)
        ser.reset_input_buffer()
        state.update(capturing=True, slot=slot, stop=threading.Event(), queue=queue.Queue())
        capture_reader_active.set()
        state["thread"] = threading.Thread(target=_ina_reader_loop, args=(state["stop"], state["queue"]), daemon=True)
        state["thread"].start()
        for cmd in ("S_TIME", "S_F INA", "S_ON"):
            ser.write((cmd + "\n").encode())
            time.sleep(0.02)
        start_btn.config(state="disabled")
        stop_btn.config(state="normal")
        status_var.set(f"Capturando {slot[0]} {slot[1]} (canal {channel})...")

    def finish_stop():
        state["stop"].set()
        state["thread"].join(timeout=1)
        capture_reader_active.clear()
        slot = state["slot"]
        state["capturing"] = False
        refresh_row(slot)
        start_btn.config(state="normal")
        status_var.set(f"Toma de {slot[0]} {slot[1]} terminada: {len(ina_captures[slot])} muestras.")

    def stop():
        if not state["capturing"]:
            return
        ser.write(b"S_OFF\n")
        stop_btn.config(state="disabled")
        root.after(300, finish_stop)  # deja llegar las ultimas lineas del stream

    def poll():
        q = state["queue"]
        got_sample = False
        if q is not None:
            while True:
                try:
                    kind, payload = q.get_nowait()
                except queue.Empty:
                    break
                if kind == "D" and state["capturing"]:
                    ina_captures[state["slot"]].append(payload)
                    t_ms, i_ma, v_bus, p_mw = payload
                    if not plot_buf["t"]:
                        plot_buf["_t0"] = t_ms
                    plot_buf["t"].append((t_ms - plot_buf.get("_t0", t_ms)) / 1000.0)
                    plot_buf["i"].append(i_ma)
                    plot_buf["v"].append(v_bus)
                    plot_buf["p"].append(p_mw)
                    got_sample = True
                    last_var.set("Ultima lectura: {:.1f} mA | {:.2f} V | {:.0f} mW".format(i_ma, v_bus, p_mw))
                elif kind == "L":
                    status_var.set(f"Firmware: {payload}")
            if state["capturing"]:
                refresh_row(state["slot"])
                if got_sample:
                    slot = state["slot"]
                    update_plot(list(plot_buf["t"]), list(plot_buf["i"]), list(plot_buf["v"]), list(plot_buf["p"]),
                                title=f"{slot[0]} {slot[1]} (en vivo)")
        root.after(50, poll)

    def write_rows(writer, slot, rows):
        channel = LEG_JOINT_CHANNELS[slot[0]][slot[1]]
        for r in rows:
            writer.writerow([slot[0], slot[1], channel, *r])

    def export(slots, default_name):
        slots = [s for s in slots if ina_captures.get(s)]
        if not slots:
            messagebox.showinfo("Exportar", "No hay datos capturados para exportar.")
            return
        path = filedialog.asksaveasfilename(parent=root, defaultextension=".csv", initialfile=default_name, filetypes=[("CSV", "*.csv")])
        if not path:
            return
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["leg", "joint", "channel", *INA_COLUMNS])
            for slot in slots:
                write_rows(writer, slot, ina_captures[slot])
        status_var.set(f"Exportado: {path}")

    def export_selected():
        slot, _ = current_slot()
        export([slot], f"ina_{slot[0]}_{slot[1]}.csv")

    def export_all():
        export([(leg, joint) for leg, joint, _ in SERVO_SLOTS], "ina_todos_los_servos.csv")

    def on_close():
        if state["capturing"]:
            try:
                ser.write(b"S_OFF\n")
            except Exception:
                pass
            state["stop"].set()
            capture_reader_active.clear()
            state["capturing"] = False
        root.destroy()

    start_btn.config(command=start)
    stop_btn.config(command=stop)
    export_sel_btn.config(command=export_selected)
    export_all_btn.config(command=export_all)
    root.protocol("WM_DELETE_WINDOW", on_close)
    poll()
    root.mainloop()


def open_ina_popup():
    global _ina_popup_thread
    if _ina_popup_thread is not None and _ina_popup_thread.is_alive():
        print("La ventana de captura INA ya esta abierta.")
        return
    _ina_popup_thread = threading.Thread(target=_run_ina_popup, daemon=True)
    _ina_popup_thread.start()


open_ina_popup()

## Otros comandos generales del firmware

In [ ]:
print(send_raw("INFO"))

In [ ]:
# LED de la placa
print(send_raw("ON"))
time.sleep(1)
print(send_raw("OFF"))

## Cerrar el puerto

In [26]:
ser.close()